In [20]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

In [21]:
def make_table(table):
    header=[i.text for i in table.find(lambda tag: tag.name=='tr' and tag.find('th')).find_all('th')]
    data=[[cell.text.strip() for cell in row if cell.text.strip()!='']
    for row in table.find_all(lambda tag: tag.name=='tr' and tag.find('td'))]
    df=pd.DataFrame(data, columns=header)
    return df

In [22]:
def get_qsp_stat(soup):
    table=soup.find(attrs={'data-testid': re.compile('qsp-statistics')})
    df=make_table(table)
    return df

In [23]:
def get_stats_highlight(soup):
    table=soup.find(attrs={'data-testid': re.compile('stats-highlight')})
    df=pd.DataFrame([[i.text.strip() for i in row.find_all('td')] 
                    for row in table.find_all('tr')])
    return df

In [32]:
def get_response(url):
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36 Edg/128.0.0.0'}
    r=requests.get(url, headers=headers)
    return r

In [24]:
def get_key_stat(ticker):
    url=f'https://finance.yahoo.com/quote/{ticker}/key-statistics/'
    r=get_response(url)
    soup=BeautifulSoup(r.content)
    qsp_stat=get_qsp_stat(soup)
    stats_highlight=get_stats_highlight(soup)
    return qsp_stat, stats_highlight

In [28]:
ticker='0005.hk'
dfs=get_key_stat(ticker)

In [33]:
url='https://finance.yahoo.com/quote/NVDA/balance-sheet/'
r=get_response(url)
r

<Response [200]>

In [80]:
soup=BeautifulSoup(r.content)
tables=soup.find_all(attrs={'class': re.compile('tableContainer')})
data_table=[[cell.get_text().strip() for cell in 
             row.find_all('div', {'class': re.compile('column')})] 
             for row in tables[0].find_all('div', {'class': re.compile('row')})
             if row.find_all('div', {'class': re.compile('column')})]
pd.DataFrame(data_table)

,0,1,2,3,4,5
0,Breakdown,1/31/2024,1/31/2023,1/31/2022,1/31/2021,1/31/2020
1,Total Assets,"65,728,000.00","41,182,000.00","44,187,000.00","28,791,000.00",--
2,Total Liabilities Net Minority Interest,"22,750,000.00","19,081,000.00","17,575,000.00","11,898,000.00",--
3,Total Equity Gross Minority Interest,"42,978,000.00","22,101,000.00","26,612,000.00","16,893,000.00",--
4,Total Capitalization,"51,437,000.00","31,804,000.00","37,558,000.00","22,857,000.00",--
5,Common Stock Equity,"42,978,000.00","22,101,000.00","26,612,000.00","16,893,000.00",--
6,Capital Lease Obligations,"1,347,000.00","1,078,000.00","885,000.00","634,000.00",--
7,Net Tangible Assets,"37,436,000.00","16,053,000.00","19,924,000.00","9,963,000.00",--
8,Working Capital,"33,714,000.00","16,510,000.00","24,494,000.00","12,130,000.00",--
9,Invested Capital,"52,687,000.00","33,054,000.00","37,558,000.00","23,856,000.00",--


In [ ]:
.find(lambda tag: tag.name=='tr' and tag.find('th'))